# EWS Fraud Detection - Tahap 6A: Inventarisasi, Deteksi Bagian, & Pembersihan Teks Annual Report

Notebook ini berfungsi untuk melakukan inventarisasi seluruh berkas **Annual Report (Laporan Tahunan)** yang berada di folder `Laporan_Tahunan` di bawah folder emiten IDX Downloads, melakukan klasifikasi deteksi bagian (*Section Detection*), melakukan ekstraksi teks spasial, serta menerapkan aturan pembersihan data teks.

### Rangkaian Pekerjaan:
1. **PDF Inventory**: Pembuatan daftar inventaris berkas PDF Laporan Tahunan beserta ukurannya.
2. **Metadata Merge**: Menggabungkan data inventaris berkas dengan data nama perusahaan dan sektor bisnis dari `Rekap_Perusahaan_2021_2024.xlsx`.
3. **Section Detection Logic**: Heuristik pencarian halaman pembatas bab (*separator page*) menggunakan batas kerapatan karakter rendah (< 450) dan pencarian sub-bagian untuk **Management Discussion & Analysis (MDA)**, **Prospek Usaha**, **Risiko Usaha**, dan **Tata Kelola**.
4. **Spatial Text Extraction**: Membatasi margin atas dan bawah (Y-clipping) sebesar 60 poin untuk otomatis mengeliminasi running Header, Footer, dan Nomor Halaman secara bawaan.
5. **Data Cleaning Engine**: Normalisasi teks (lowercase), penghilangan URL, email, angka berlebihan/tabuler, dan spasi ganda untuk keperluan analisis teks NLP.

In [28]:
# Installations cell (uncomment if running in Google Colab / new environment)
# !pip install pymupdf pytesseract pillow pandas openpyxl

In [29]:
import os
import io
import re
import shutil
import tempfile
import pandas as pd
import fitz  # PyMuPDF
from PIL import Image
import pytesseract
from tqdm import tqdm

# Konfigurasi path Tesseract jika diperlukan (Windows)
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

base_path = r"G:\My Drive\Dataset PUI-PT\IDX_Downloads"
rekap_path = r"../Data_Crawling/Rekap_Perusahaan_2021_2024.xlsx"
out_inventory_path = "Annual_Report_Inventory.xlsx"

## 1. Memindai Direktori & Membangun Inventaris Laporan Tahunan

In [30]:
print("Memulai pemindaian berkas Laporan Tahunan...")
records = []

for root, dirs, files in os.walk(base_path):
    if os.path.basename(root) == 'Laporan_Tahunan':
        parts = os.path.relpath(root, base_path).split(os.sep)
        if len(parts) >= 2:
            ticker = parts[0].strip().upper()
            year = parts[1].strip()
            
            for file in files:
                if file.lower().endswith('.pdf'):
                    file_path = os.path.join(root, file)
                    try:
                        size_mb = os.path.getsize(file_path) / (1024 * 1024)
                    except:
                        size_mb = 0
                    records.append({
                        'kode': ticker,
                        'tahun': year,
                        'file': file,
                        'size_mb': size_mb,
                        'path': file_path
                    })

df_files = pd.DataFrame(records)
print(f"Pemindaian selesai. Menemukan {len(df_files)} berkas PDF.")

Memulai pemindaian berkas Laporan Tahunan...
Pemindaian selesai. Menemukan 5381 berkas PDF.


## 2. Menggabungkan Nama Perusahaan & Sektor Bisnis

In [31]:
if os.path.exists(rekap_path):
    df_rekap = pd.read_excel(rekap_path)
    
    # Standardisasi join key
    df_rekap['kode_clean'] = df_rekap['Kode'].astype(str).str.strip().str.upper()
    df_rekap['Nama Perusahaan'] = df_rekap['Nama Perusahaan'].astype(str).str.replace(r'[\r\n\t]+', ' ', regex=True).str.strip()
    df_rekap['Sektor'] = df_rekap['Sektor'].astype(str).str.replace(r'[\r\n\t]+', ' ', regex=True).str.strip()
    
    df_rekap_sub = df_rekap[['kode_clean', 'Nama Perusahaan', 'Sektor']].drop_duplicates()
    
    df_files['kode_clean'] = df_files['kode']
    df_merged = df_files.merge(df_rekap_sub, on='kode_clean', how='left')
    df_merged.drop(columns=['kode_clean'], inplace=True)
    
    # Susun kolom
    cols = ['kode', 'Nama Perusahaan', 'Sektor', 'tahun', 'file', 'size_mb', 'path']
    df_merged = df_merged[[c for c in cols if c in df_merged.columns]]
    
    df_merged.to_excel(out_inventory_path, index=False)
    print(f"Inventaris berhasil disimpan ke '{out_inventory_path}'. Shape: {df_merged.shape}")
    display(df_merged.head(5))
else:
    print(f"ERROR: File rekap tidak ditemukan di {rekap_path}")

Inventaris berhasil disimpan ke 'Annual_Report_Inventory.xlsx'. Shape: (5397, 7)


,kode,Nama Perusahaan,Sektor,tahun,file,size_mb,path
0,AALI,Astra Agro Lestari Tbk,Consumer Non-Cyclicals,2022,AnnualReport2022-AALI-att1.pdf,7.931753,G:\My Drive\Dataset PUI-PT\IDX_Downloads\AALI\...
1,AALI,Astra Agro Lestari Tbk,Consumer Non-Cyclicals,2023,AnnualReport2023-AALI-att1.pdf,4.586615,G:\My Drive\Dataset PUI-PT\IDX_Downloads\AALI\...
2,AALI,Astra Agro Lestari Tbk,Consumer Non-Cyclicals,2023,AnnualReport2023-AALI.pdf,0.005309,G:\My Drive\Dataset PUI-PT\IDX_Downloads\AALI\...
3,AALI,Astra Agro Lestari Tbk,Consumer Non-Cyclicals,2024,AnnualReport2024-AALI-att1.pdf,9.543583,G:\My Drive\Dataset PUI-PT\IDX_Downloads\AALI\...
4,AALI,Astra Agro Lestari Tbk,Consumer Non-Cyclicals,2024,AnnualReport2024-AALI.pdf,0.005443,G:\My Drive\Dataset PUI-PT\IDX_Downloads\AALI\...


## 3. Implementasi Pembersihan Teks (Cleaning Engine)

In [32]:
def clean_extracted_text(text):
    """
    Membersihkan teks dari URL, email, angka berlebihan/tabel, 
    serta melakukan standardisasi spasi dan lowercase.
    """
    text = text.lower()
    
    # 1. Hilangkan URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    
    # 2. Hilangkan Email
    text = re.sub(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b', ' ', text)
    
    # 3. Hilangkan Angka berlebihan / tabel angka
    text = re.sub(r'\b\d+[\d.,%/-]*\b', ' ', text)
    
    # 4. Hilangkan spasi berulang dan newlines
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

## 4. Implementasi Section Detection & Spatial Extraction Engine

In [33]:
def extract_sections_from_pdf(pdf_path):
    """
    Deteksi bagian MDA, Prospek, Risiko, dan Tata Kelola pada PDF.
    Menyalin file secara lokal terlebih dahulu untuk menghindari read error pada virtual drive.
    """
    if not os.path.exists(pdf_path):
        return None
        
    temp_dir = tempfile.gettempdir()
    temp_pdf = os.path.join(temp_dir, f"temp_section_extract_{os.path.basename(pdf_path)}")
    
    try:
        shutil.copy2(pdf_path, temp_pdf)
        doc = fitz.open(temp_pdf)
        num_pages = len(doc)
        
        # Definisi Kata Kunci Pencarian Separator Bab
        sections_def = {
            'mda': ["analisis dan pembahasan manajemen", "management discussion and analysis", "management discussion & analysis"],
            'governance': ["tata kelola perusahaan", "corporate governance", "good corporate governance", "laporan tata kelola"],
            'csr': ["tanggung jawab sosial perusahaan", "corporate social responsibility", "laporan tanggung jawab sosial"],
            'fs': ["laporan keuangan konsolidasian", "consolidated financial statements", "laporan keuangan"]
        }
        
        detected_starts = {k: None for k in sections_def}
        
        # Fase 1: Deteksi halaman pemisah bab (Kerapatan Teks Rendah < 450 karakter)
        for i in range(15, num_pages):
            page = doc[i]
            rect = page.rect
            clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
            text = page.get_text("text", clip=clip_rect)
            text_clean = " ".join([w.strip() for w in text.lower().split() if w.strip()])
            
            if len(text_clean) < 450:
                for sec, kws in sections_def.items():
                    if detected_starts[sec] is not None:
                        continue
                    for kw in kws:
                        kw_clean = " ".join(kw.split())
                        if kw_clean in text_clean:
                            detected_starts[sec] = i
                            break
                    if detected_starts[sec] is not None:
                        break
                        
        # Fase 2: Fallback (Pencarian Tanpa Batas Kerapatan Teks)
        for sec in ['mda', 'governance', 'csr', 'fs']:
            if detected_starts[sec] is None:
                search_start = 15
                if sec == 'governance' and detected_starts['mda'] is not None:
                    search_start = detected_starts['mda']
                elif sec == 'csr' and detected_starts['governance'] is not None:
                    search_start = detected_starts['governance']
                elif sec == 'fs' and detected_starts['csr'] is not None:
                    search_start = detected_starts['csr']
                    
                for i in range(search_start, num_pages):
                    page = doc[i]
                    rect = page.rect
                    clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
                    text = page.get_text("text", clip=clip_rect)
                    text_clean = " ".join(text.lower().split())
                    
                    found = False
                    for kw in sections_def[sec]:
                        kw_clean = " ".join(kw.split())
                        if kw_clean in text_clean:
                            detected_starts[sec] = i
                            found = True
                            break
                    if found:
                        break
                        
        # Penyesuaian nilai default jika bab tidak terdeteksi sama sekali
        if detected_starts['mda'] is None:
            detected_starts['mda'] = 15
        if detected_starts['governance'] is None:
            detected_starts['governance'] = min(detected_starts['mda'] + 30, num_pages - 1)
        if detected_starts['csr'] is None:
            detected_starts['csr'] = min(detected_starts['governance'] + 80, num_pages - 1)
        if detected_starts['fs'] is None:
            detected_starts['fs'] = min(detected_starts['csr'] + 10, num_pages - 1)
            
        # Rentang Halaman Utama
        mda_start_idx = detected_starts['mda']
        mda_end_idx = detected_starts['governance'] - 1
        
        gcg_start_idx = detected_starts['governance']
        gcg_end_idx = detected_starts['csr'] - 1
        
        # Deteksi Sub-Bagian: Prospek Usaha (Di dalam rentang MDA)
        prospect_start_idx = None
        for i in range(mda_start_idx, mda_end_idx + 1):
            page = doc[i]
            rect = page.rect
            clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
            text_clean = " ".join(page.get_text("text", clip=clip_rect).lower().split())
            if "prospek" in text_clean or "outlook" in text_clean:
                prospect_start_idx = i
                break
        if prospect_start_idx is None:
            prospect_start_idx = max(mda_start_idx, mda_end_idx - 5)
            
        prospect_end_idx = mda_end_idx
        mda_general_end_idx = prospect_start_idx - 1
        
        # Deteksi Sub-Bagian: Risiko Usaha (Di dalam rentang Tata Kelola)
        risk_start_idx = None
        for i in range(gcg_start_idx, gcg_end_idx + 1):
            page = doc[i]
            rect = page.rect
            clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
            text_clean = " ".join(page.get_text("text", clip=clip_rect).lower().split())
            if "manajemen risiko" in text_clean or "risiko usaha" in text_clean or "faktor risiko" in text_clean:
                risk_start_idx = i
                break
        if risk_start_idx is None:
            risk_start_idx = max(gcg_start_idx, gcg_end_idx - 10)
        risk_end_idx = min(risk_start_idx + 5, gcg_end_idx) # Ambil 6 halaman sub-bab risiko
        
        # Fungsi Pembantu Ekstraksi dengan Spatial Clipping
        def get_range_text(start, end):
            txt_list = []
            for idx in range(start, end + 1):
                page = doc[idx]
                rect = page.rect
                # Hapus margin atas/bawah 60pt untuk memotong Header/Footer/Nomor Halaman
                clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
                p_text = page.get_text("text", clip=clip_rect).strip()
                if len(p_text) < 50:
                    p_text = " "
                txt_list.append(p_text)
            return "\n".join(txt_list)
            
        mda_raw = get_range_text(mda_start_idx, mda_general_end_idx)
        prospect_raw = get_range_text(prospect_start_idx, prospect_end_idx)
        risk_raw = get_range_text(risk_start_idx, risk_end_idx)
        governance_raw = get_range_text(gcg_start_idx, gcg_end_idx)
        
        doc.close()
        if os.path.exists(temp_pdf):
            os.remove(temp_pdf)
            
        # Bersihkan masing-masing teks
        return {
            'mda_text': clean_extracted_text(mda_raw),
            'prospect_text': clean_extracted_text(prospect_raw),
            'risk_text': clean_extracted_text(risk_raw),
            'governance_text': clean_extracted_text(governance_raw)
        }
    except Exception as e:
        if os.path.exists(temp_pdf):
            try:
                os.remove(temp_pdf)
            except:
                pass
        print(f"Error processing {pdf_path}: {str(e)}")
        return None

## 5. Demonstrasi & Validasi Hasil Ekstraksi & Pembersihan

In [34]:
# Demonstrasi pada sampel emiten AALI 2024
if 'df_merged' in locals() and len(df_merged) > 0:
    # Cari index untuk AALI 2024
    sample_idx_list = df_merged[(df_merged['kode'] == 'AALI') & (df_merged['tahun'] == 2024)].index.tolist()
    if not sample_idx_list:
        sample_idx_list = df_merged[(df_merged['kode'] == 'AALI') & (df_merged['tahun'] == '2024')].index.tolist()
        
    if sample_idx_list:
        row = df_merged.loc[sample_idx_list[0]]
        print(f"Menguji pada: {row['kode']} - {row['tahun']} ({row['file']})")
        print(f"Path: {row['path']}")
        
        res = extract_sections_from_pdf(row['path'])
        if res:
            for k, v in res.items():
                print(f"\n========================================")
                print(f"BAGIAN: {k} (Panjang Teks = {len(v)} karakter)")
                print(f"========================================")
                print(v[:300] + "...")
        else:
            print("Gagal mengekstrak bagian teks sampel.")
    else:
        print("AALI 2024 tidak ditemukan di data inventaris.")
else:
    print("Daftar inventaris kosong atau tidak dapat diakses.")

Menguji pada: AALI - 2024 (AnnualReport2024-AALI-att1.pdf)
Path: G:\My Drive\Dataset PUI-PT\IDX_Downloads\AALI\2024\Laporan_Tahunan\AnnualReport2024-AALI-att1.pdf

BAGIAN: mda_text (Panjang Teks = 63661 karakter)
analisis dan pembahasan manajemen management discussion and analysis tinjauan industri industry overview pada tahun , produksi minyak nabati mengalami penurunan yang utamanya disebabkan oleh minyak kelapa sawit, minyak bunga matahari, maupun minyak rapeseed. hal ini membuat harga minyak nabati menga...

BAGIAN: prospect_text (Panjang Teks = 24458 karakter)
kebijakan keuangan dibentuknya kebijakan keuangan oleh perseroan ditujukan untuk membantu mengurangi risiko dari segi valuta asing maupun tingkat suku bunga yang fluktuatif. pengelolaan dana yang dimiliki selalu dikelola oleh perseroan untuk meminimalisir tingkat risiko dan menghindari potensi kerug...

BAGIAN: risk_text (Panjang Teks = 21843 karakter)
direksi board of directors memenuhi maksud dan tujuan perseroan berdasark

## 6. Pembuatan Fitur Baseline NLP (Sentiment, Readability, & Keyword Frequency)

Pada tahap ini, kita mengekstrak fitur baseline NLP untuk seluruh **5.381 berkas PDF** (total 5.397 baris pada inventaris) secara langsung di dalam notebook.

Mekanisme ini sangat robust dan disesuaikan untuk dijalankan di Google Drive Desktop:
1. **Google Drive Desktop Sync Waiter**: Menyalin file ke temp lokal Windows dan menunggu hingga download penuh selesai (15x coba, total 45 detik).
2. **Penyimpanan Checkpoint Atomik**: Checkpoint disimpan setiap 10 records untuk mencegah korupsi file jika notebook dihentikan.
3. **Kemampuan Resume Otomatis**: Secara otomatis memuat progress dari `processed_indices.txt` dan melanjutkan pemrosesan.

In [35]:
# 1. Definisi Kamus Sentimen Finansial & Fungsi Pembantu Keterbacaan
import os
import re
import fitz
import shutil
import tempfile
import time
import pandas as pd

positive_words = {"untung", "laba", "naik", "positif", "tumbuh", "meningkat", "optimis", "baik", "sukses", "ekspansi", "efisiensi", "membaik", "penguatan"}
negative_words = {"rugi", "turun", "negatif", "menurun", "pesimis", "buruk", "gagal", "risiko", "tantangan", "tekanan", "lambat", "ketidakpastian", "kerugian", "penurunan", "melemah"}
target_kws = ["risiko", "tantangan", "ketidakpastian", "pertumbuhan", "restrukturisasi", "kerugian"]

def calculate_sentiment(text):
    words = text.split()
    if not words:
        return 0.0
    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)
    total = pos_count + neg_count
    if total == 0:
        return 0.0
    return (pos_count - neg_count) / total

def calculate_readability_lix(raw_text):
    raw_text = re.sub(r'\s+', ' ', raw_text)
    words = [w.strip(',.()[]{}":;') for w in raw_text.split() if w.strip()]
    if not words:
        return 0.0
    word_count = len(words)
    sentences = [s for s in re.split(r'[.!?]+', raw_text) if s.strip()]
    sentence_count = max(1, len(sentences))
    long_words = sum(1 for w in words if len(w) > 6)
    return (word_count / sentence_count) + (long_words * 100 / word_count)

def robust_copy_and_wait(src, dst):
    if not os.path.exists(src):
        return False
    try:
        orig_size = os.path.getsize(src)
    except:
        return False
    if orig_size == 0:
        try:
            shutil.copy2(src, dst)
            return True
        except:
            return False
    for attempt in range(15):
        try:
            if os.path.exists(dst):
                try: os.remove(dst)
                except: pass
            shutil.copy2(src, dst)
            for wait_sec in range(10):
                if os.path.exists(dst):
                    if os.path.getsize(dst) == orig_size:
                        return True
                time.sleep(1.0)
        except:
            time.sleep(3.0)
    return False

In [36]:
# 2. Fungsi Utama Ekstraksi Fitur & Penyelamatan Checkpoint
def process_single_pdf(row_tuple):
    idx, row = row_tuple
    pdf_path = row['path']
    kode = row['kode']
    tahun = row['tahun']
    size_mb = row['size_mb']
    
    temp_dir = tempfile.gettempdir()
    temp_pdf = os.path.join(temp_dir, f"temp_nb_all_feat_idx_{idx}_{os.path.basename(pdf_path)}")
    
    copied = robust_copy_and_wait(pdf_path, temp_pdf)
    if not copied:
        return {
            'idx': idx, 'Kode': kode, 'Nama perusahaan': row['Nama Perusahaan'], 'sektor': row['Sektor'], 'tahun': tahun,
            'sentiment': 0.0, 'risk_words': 0, 'readability': 0.0, 'text_length': 0
        }
        
    try:
        doc = fitz.open(temp_pdf)
        num_pages = len(doc)
        
        sections_def = {
            'mda': ["analisis dan pembahasan manajemen", "management discussion and analysis", "management discussion & analysis"],
            'governance': ["tata kelola perusahaan", "corporate governance", "good corporate governance", "laporan tata kelola"],
            'csr': ["tanggung jawab sosial perusahaan", "corporate social responsibility", "laporan tanggung jawab sosial"],
            'fs': ["laporan keuangan konsolidasian", "consolidated financial statements", "laporan keuangan"]
        }
        detected_starts = {k: None for k in sections_def}
        for i in range(15, num_pages):
            page = doc[i]
            rect = page.rect
            clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
            text = page.get_text("text", clip=clip_rect)
            text_clean = " ".join([w.strip() for w in text.lower().split() if w.strip()])
            if len(text_clean) < 450:
                for sec, kws in sections_def.items():
                    if detected_starts[sec] is not None:
                        continue
                    for kw in kws:
                        kw_clean = " ".join(kw.split())
                        if kw_clean in text_clean:
                            detected_starts[sec] = i
                            break
                    if detected_starts[sec] is not None:
                        break
                        
        for sec in ['mda', 'governance', 'csr', 'fs']:
            if detected_starts[sec] is None:
                search_start = 15
                if sec == 'governance' and detected_starts['mda'] is not None:
                    search_start = detected_starts['mda']
                elif sec == 'csr' and detected_starts['governance'] is not None:
                    search_start = detected_starts['governance']
                elif sec == 'fs' and detected_starts['csr'] is not None:
                    search_start = detected_starts['csr']
                for i in range(search_start, num_pages):
                    page = doc[i]
                    rect = page.rect
                    clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
                    text_clean = " ".join(page.get_text("text", clip=clip_rect).lower().split())
                    found = False
                    for kw in sections_def[sec]:
                        kw_clean = " ".join(kw.split())
                        if kw_clean in text_clean:
                            detected_starts[sec] = i
                            found = True
                            break
                    if found:
                        break
                        
        if detected_starts['mda'] is None: detected_starts['mda'] = 15
        if detected_starts['governance'] is None: detected_starts['governance'] = min(detected_starts['mda'] + 30, num_pages - 1)
        if detected_starts['csr'] is None: detected_starts['csr'] = min(detected_starts['governance'] + 80, num_pages - 1)
        if detected_starts['fs'] is None: detected_starts['fs'] = min(detected_starts['csr'] + 10, num_pages - 1)
        
        mda_start = detected_starts['mda']
        mda_end = detected_starts['governance'] - 1
        gcg_start = detected_starts['governance']
        gcg_end = detected_starts['csr'] - 1
        
        # Subsection: Prospect & Risk
        prospect_start = None
        for i in range(mda_start, mda_end + 1):
            page = doc[i]
            rect = page.rect
            clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
            text_clean = " ".join(page.get_text("text", clip=clip_rect).lower().split())
            if "prospek" in text_clean or "outlook" in text_clean:
                prospect_start = i
                break
        if prospect_start is None: prospect_start = max(mda_start, mda_end - 5)
        prospect_end = mda_end
        mda_general_end = prospect_start - 1
        
        risk_start = None
        for i in range(gcg_start, gcg_end + 1):
            page = doc[i]
            rect = page.rect
            clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
            text_clean = " ".join(page.get_text("text", clip=clip_rect).lower().split())
            if "manajemen risiko" in text_clean or "risiko usaha" in text_clean or "faktor risiko" in text_clean:
                risk_start = i
                break
        if risk_start is None: risk_start = max(gcg_start, gcg_end - 10)
        risk_end = min(risk_start + 5, gcg_end)
        
        def get_text_range(start, end):
            txt = []
            for idx_p in range(start, end + 1):
                page = doc[idx_p]
                rect = page.rect
                clip_rect = fitz.Rect(rect.x0, rect.y0 + 60, rect.x1, rect.y1 - 60)
                p_txt = page.get_text("text", clip=clip_rect).strip()
                if len(p_txt) < 50: p_txt = " "
                txt.append(p_txt)
            return "\n".join(txt)
            
        mda_raw = get_text_range(mda_start, mda_general_end)
        prospect_raw = get_text_range(prospect_start, prospect_end)
        risk_raw = get_text_range(risk_start, risk_end)
        gov_raw = get_text_range(gcg_start, gcg_end)
        
        doc.close()
        if os.path.exists(temp_pdf): os.remove(temp_pdf)
        
        raw_combined = mda_raw + "\n" + prospect_raw + "\n" + risk_raw + "\n" + gov_raw
        readability = calculate_readability_lix(raw_combined)
        
        mda_clean = clean_extracted_text(mda_raw)
        prospect_clean = clean_extracted_text(prospect_raw)
        risk_clean = clean_extracted_text(risk_raw)
        gov_clean = clean_extracted_text(gov_raw)
        
        clean_combined = mda_clean + " " + prospect_clean + " " + risk_clean + " " + gov_clean
        text_length = len(clean_combined)
        sentiment = calculate_sentiment(clean_combined)
        
        counts = {kw: clean_combined.count(kw) for kw in target_kws}
        risk_words = counts["risiko"] + counts["tantangan"] + counts["ketidakpastian"] + counts["restrukturisasi"] + counts["kerugian"]
        
        return {
            'idx': idx, 'Kode': kode, 'Nama perusahaan': row['Nama Perusahaan'], 'sektor': row['Sektor'], 'tahun': tahun,
            'sentiment': sentiment, 'risk_words': risk_words, 'readability': readability, 'text_length': text_length
        }
    except:
        if os.path.exists(temp_pdf):
            try: os.remove(temp_pdf)
            except: pass
        return {
            'idx': idx, 'Kode': kode, 'Nama perusahaan': row['Nama Perusahaan'], 'sektor': row['Sektor'], 'tahun': tahun,
            'sentiment': 0.0, 'risk_words': 0, 'readability': 0.0, 'text_length': 0
        }

def save_and_publish_progress(processed_results, df_original, checkpoint_path, final_output_path):
    df_progress = pd.DataFrame(processed_results)
    processed_indices = set(df_progress['idx'].values)
    missing_records = []
    for idx, row in df_original.iterrows():
        if idx not in processed_indices:
            missing_records.append({
                'idx': idx, 'Kode': row['kode'], 'Nama perusahaan': row['Nama Perusahaan'], 'sektor': row['Sektor'], 'tahun': row['tahun'],
                'sentiment': 0.0, 'risk_words': 0, 'readability': 0.0, 'text_length': 0
            })
    if missing_records:
        df_combined = pd.concat([df_progress, pd.DataFrame(missing_records)], ignore_index=True)
    else:
        df_combined = df_progress
    df_combined = df_combined.sort_values(by='idx').reset_index(drop=True)
    
    temp_dir = tempfile.gettempdir()
    temp_xlsx = os.path.join(temp_dir, f"temp_nb_progress_{int(time.time())}.xlsx")
    df_combined.to_excel(temp_xlsx, index=False)
    try:
        shutil.copy2(temp_xlsx, checkpoint_path)
        shutil.copy2(temp_xlsx, final_output_path)
    except Exception as e:
        print(f"[ERROR_SAVE] {e}")
    try: os.remove(temp_xlsx)
    except: pass

In [37]:
# 3. Blok Eksekusi Utama (Dapat dijalankan langsung di notebook)
from concurrent.futures import ThreadPoolExecutor, as_completed

checkpoint_path = "Annual_Report_Features_Checkpoint.xlsx"
final_output_path = "Annual_Report_Features.xlsx"
processed_txt_path = "processed_indices.txt"

# Pastikan folder dasar pemrosesan benar
if 'df_merged' in locals() and len(df_merged) > 0:
    df_original = df_merged.copy()
    df_original['idx'] = df_original.index
    total_records = len(df_original)
    print(f"Total inventory records: {total_records}")
    
    # Pelacakan indeks yang sudah diproses
    processed_indices = set()
    processed_results = []
    
    if os.path.exists(processed_txt_path):
        with open(processed_txt_path, 'r') as f:
            for line in f:
                val = line.strip()
                if val:
                    processed_indices.add(int(val))
        print(f"Memuat {len(processed_indices)} indeks yang sudah selesai dari processed_indices.txt.")
        
        if os.path.exists(checkpoint_path):
            try:
                df_ckpt = pd.read_excel(checkpoint_path)
                df_ckpt_processed = df_ckpt[df_ckpt['idx'].isin(processed_indices)]
                processed_results = df_ckpt_processed.to_dict('records')
            except Exception as e:
                print(f"Gagal memuat baris dari checkpoint: {e}")
                
    to_process = []
    for idx, row in df_original.iterrows():
        if idx not in processed_indices:
            to_process.append((idx, row))
            
    # Urutkan sisa tugas berdasarkan ukuran file terkecil terlebih dahulu
    to_process.sort(key=lambda item: item[1]['size_mb'])
    print(f"Sisa file yang harus diekstrak: {len(to_process)}")
    
    if len(to_process) > 0:
        print("Memulai extraction pool dengan 4 workers...")
        start_time = time.time()
        count = 0
        with ThreadPoolExecutor(max_workers=4) as pool:
            futures = {pool.submit(process_single_pdf, item): item for item in to_process}
            for fut in as_completed(futures):
                res = fut.result()
                processed_results.append(res)
                processed_indices.add(res['idx'])
                count += 1
                
                with open(processed_txt_path, 'a') as f:
                    f.write(f"{res['idx']}\n")
                    
                if count % 10 == 0 or count == len(to_process):
                    elapsed = time.time() - start_time
                    avg_time = elapsed / count
                    est_rem = avg_time * (len(to_process) - count)
                    print(f"Diproses {count}/{len(to_process)}. Waktu berjalan: {elapsed:.1f}s. Sisa estimasi: {est_rem:.1f}s.")
                    save_and_publish_progress(processed_results, df_original, checkpoint_path, final_output_path)
                    
        print("Seluruh ekstraksi fitur baseline NLP selesai!")
    else:
        print("Semua data sudah selesai diproses!")
        save_and_publish_progress(processed_results, df_original, checkpoint_path, final_output_path)
else:
    print("Variabel df_merged tidak terdefinisi. Jalankan sel-sel sebelumnya terlebih dahulu.")

Total inventory records: 5397
Sisa file yang harus diekstrak: 5397
Memulai extraction pool dengan 4 workers...
Diproses 10/5397. Waktu berjalan: 0.2s. Sisa estimasi: 128.9s.
Diproses 20/5397. Waktu berjalan: 1.2s. Sisa estimasi: 321.7s.
Diproses 30/5397. Waktu berjalan: 66.8s. Sisa estimasi: 11956.1s.
Diproses 40/5397. Waktu berjalan: 72.6s. Sisa estimasi: 9727.9s.
Diproses 50/5397. Waktu berjalan: 77.9s. Sisa estimasi: 8326.3s.
Diproses 60/5397. Waktu berjalan: 79.0s. Sisa estimasi: 7030.1s.
Diproses 70/5397. Waktu berjalan: 80.2s. Sisa estimasi: 6101.8s.
Diproses 80/5397. Waktu berjalan: 82.9s. Sisa estimasi: 5507.5s.
Diproses 90/5397. Waktu berjalan: 84.0s. Sisa estimasi: 4951.5s.
Diproses 100/5397. Waktu berjalan: 85.1s. Sisa estimasi: 4510.3s.
Diproses 110/5397. Waktu berjalan: 91.7s. Sisa estimasi: 4407.7s.
Diproses 120/5397. Waktu berjalan: 93.0s. Sisa estimasi: 4090.7s.
Diproses 130/5397. Waktu berjalan: 94.4s. Sisa estimasi: 3824.9s.
Diproses 140/5397. Waktu berjalan: 95.7s. S